# MLP Classifier on MNIST: Warm-Start Training & Model Capacity

Training a multi-layer perceptron on MNIST with scikit-learn, experimenting with warm-start (incremental) training schedules, and probing how much a neural network can "cheat" when a label-like signal is planted directly in the input.

In [25]:
from IPython.display import Math
from IPython.display import Latex

import numpy as np
import pandas as pd
import os
import struct

from sklearn.metrics import classification_report,confusion_matrix
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

----------------
----------------
----------------

## <font color='blue'> Section 0. MLP with scikit-learn: Explore outputs for the MNIST dataset </font>

This section builds an MLP classifier on the MNIST dataset with scikit-learn and explores its outputs — not just the top prediction, but the full probability distribution over all ten digit classes.

In [2]:
# Let's first load the MNIST data set.

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split


X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
y = y.astype(int)
X = ((X / 255.) - .5) * 2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=10000, random_state=2501, stratify=y)


A baseline MLP classifier is configured with a batch size and maximum iteration count chosen so that training converges.

In [ ]:
# Question 0-0
# Set arrays of batch sizes and max iterations to test with
batch_sizes = [5, 10, 32, 50, 64, 100, 128]
max_iters = [ 50, 100, 150, 200]
# Create array to capture statistics / values for each test
results = []
# Loop through the arrays of test values
for batch_size in batch_sizes:
    for max_iter in max_iters:
      # Print This Test's Parameters
      print(f"batch_size: {batch_size}, max_iter: {max_iter}")
      # Initialize MLP Classifier with this test's parameters
      mlp_mnist = MLPClassifier(verbose = True, hidden_layer_sizes=(50,), batch_size = batch_size, \
                          max_iter=max_iter, solver='sgd', activation='logistic',\
                          learning_rate = 'constant', learning_rate_init = 0.001, random_state=2501)
      # Fit MLP Classifier on test data
      mlp_mnist.fit(X_train,y_train)
      # If the training finished before the max iteration had been reached. The classifier converged.
      # Can also tell by the ending message of the training thanks to the verbose parameter set to true for the classifier
      converged = mlp_mnist.n_iter_ < max_iter
      # Get Predictions of the X_test data
      predictions = mlp_mnist.predict(X_test)
      # Calculate accuracy
      accuracy = accuracy_score(y_test, predictions)
      # Save Metrics & Values in Results array
      results.append((batch_size, max_iter, converged, accuracy))
      # Get Probabilities of Each Class
      probability_predictions = mlp_mnist.predict_proba(X_test)
      # Print if Model Converged For The Batch Size and Max Iterations and its Accuracy
      print(f"Converged: {converged} - Accuracy: {accuracy}\n")



batch_size: 5, max_iter: 50
Iteration 1, loss = 0.57226651
Iteration 2, loss = 0.28125709
Iteration 3, loss = 0.23351902
Iteration 4, loss = 0.20451488
Iteration 5, loss = 0.18349439
Iteration 6, loss = 0.16835462
Iteration 7, loss = 0.15590827
Iteration 8, loss = 0.14537654
Iteration 9, loss = 0.13708256
Iteration 10, loss = 0.13001910
Iteration 11, loss = 0.12384926
Iteration 12, loss = 0.11808123
Iteration 13, loss = 0.11304021
Iteration 14, loss = 0.10899001
Iteration 15, loss = 0.10463360
Iteration 16, loss = 0.10080266
Iteration 17, loss = 0.09756344
Iteration 18, loss = 0.09453881
Iteration 19, loss = 0.09180411
Iteration 20, loss = 0.08878815
Iteration 21, loss = 0.08613301
Iteration 22, loss = 0.08400569
Iteration 23, loss = 0.08189468
Iteration 24, loss = 0.08006683
Iteration 25, loss = 0.07769518
Iteration 26, loss = 0.07621200
Iteration 27, loss = 0.07438151
Iteration 28, loss = 0.07282857
Iteration 29, loss = 0.07113742
Iteration 30, loss = 0.06982537
Iteration 31, loss = 

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


Converged: True - Accuracy: 0.9684

batch_size: 5, max_iter: 100


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


Converged: True - Accuracy: 0.8632

batch_size: 5, max_iter: 150
Iteration 1, loss = 0.57226651


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


Converged: True - Accuracy: 0.9102

batch_size: 5, max_iter: 200


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


Converged: True - Accuracy: 0.8953

batch_size: 10, max_iter: 50


In [9]:
# Creating MLP Classifier using one of the settings that converged
batch_size = 64
max_iter = 500

mlp_mnist = MLPClassifier(verbose = True, hidden_layer_sizes=(50,), batch_size = batch_size, \
                          max_iter=max_iter, solver='sgd', activation='logistic',\
                          learning_rate = 'constant', learning_rate_init = 0.001, random_state=2501)

mlp_mnist.fit(X_train,y_train)

converged = mlp_mnist.n_iter_ < max_iter
print(converged)
#

Iteration 1, loss = 1.70994232
Iteration 2, loss = 0.95569378
Iteration 3, loss = 0.67110556
Iteration 4, loss = 0.54074227
Iteration 5, loss = 0.46856456
Iteration 6, loss = 0.42324863
Iteration 7, loss = 0.39204168
Iteration 8, loss = 0.36907797
Iteration 9, loss = 0.35130105
Iteration 10, loss = 0.33671814
Iteration 11, loss = 0.32456279
Iteration 12, loss = 0.31429788
Iteration 13, loss = 0.30533321
Iteration 14, loss = 0.29724721
Iteration 15, loss = 0.28987095
Iteration 16, loss = 0.28326889
Iteration 17, loss = 0.27717208
Iteration 18, loss = 0.27166676
Iteration 19, loss = 0.26639564
Iteration 20, loss = 0.26140157
Iteration 21, loss = 0.25671940
Iteration 22, loss = 0.25234116
Iteration 23, loss = 0.24817009
Iteration 24, loss = 0.24429060
Iteration 25, loss = 0.24042280
Iteration 26, loss = 0.23676118
Iteration 27, loss = 0.23328458
Iteration 28, loss = 0.23005543
Iteration 29, loss = 0.22673805
Iteration 30, loss = 0.22369456
Iteration 31, loss = 0.22066047
Iteration 32, los

In [10]:
predictions = mlp_mnist.predict(X_test)
probability_predictions = mlp_mnist.predict_proba(X_test)

accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy}")

Accuracy: 0.9713


A helper function extracts the second-best prediction for each point — the label with the second-highest probability in the model's softmax output.

In [11]:
# Question 0-1
def get_second_best_prediction(probabilities):
  # Returns the class label that has the second highest predicted probability
  # Sorts the class indices for each row by probability, ascending order
  probabilities_sorted = np.argsort(probabilities, axis=1)
  # Get second to last class index
  second_best = probabilities_sorted[:, -2]
  return second_best

second_best = get_second_best_prediction(probability_predictions)

For every misclassified point, this checks whether the second-best prediction would actually have been correct, and reports what percentage of the model's mistakes are "near misses" in that sense.

In [20]:
# Question 0-2
# Calculate the second best predictions for each point
second_best = get_second_best_prediction(probability_predictions)
# Create array that stores true/false based on whether the best prediction was correct
misclassified_points = predictions != y_test
# Create array that stores true/false based on whether the second best prediction was correct for only the misclassified points
second_best_correct = second_best[misclassified_points] == y_test[misclassified_points]
# Calculate Number of Misclassified Points and Number of Second Best Predictions that were Correct
num_misclassified = misclassified_points.sum()
num_second_best_correct = second_best_correct.sum()
# Calculate %
percentage_second_best_correct = num_second_best_correct / num_misclassified * 100

print("Number of misclassified points: ", num_misclassified)
print("Number of second best predictions that were correct: ", num_second_best_correct)
print("Percentage of misclassified points where the second best prediction was correct: ", f"{percentage_second_best_correct:.2f}%")

Number of misclassified points:  287
Number of second best predictions that were correct:  177
Percentage of misclassified points where the second best prediction was correct:  61.67%


After evaluating the 287 misclassified data points, we found that approximately 61.67% of them had the correct label as the second highest predicted probability. This probability indicates that many errors could be considered "near misses". The model's confidence ranking still places the true class label close to the top. This suggests the model is able to perform fairly well, even on challenging data points. I think it would be best to use this classifier by leveraging the top-k predictions instead of just the best. This could improve overall generalization.

Handwritten digits like 3 and 5 are often visually similar to humans, which raises the question of whether the MLP picks up on the same similarities. For every true label $i$, this computes how often each other label $j$ shows up as the second-best prediction among points with true label $i$ — effectively a confusion matrix built from second-best predictions instead of top predictions.

In [30]:
# Question 0-3
probability_predictions = mlp_mnist.predict_proba(X_test)
second_best = get_second_best_prediction(probability_predictions)

# Create a matrix of 0s, the size of 10x10. Each cell (i, j) represents the true label i and second best prediction j
# Use this matrix to store the number of times the true label is i and the second best prediction is j
frequency_matrix = np.zeros((10, 10), dtype=float)
# Loop Through y_test and second_best predictions, for each (i, j) combination increment count by 1
for i, j in zip(y_test, second_best):
  frequency_matrix[i, j] += 1
# Get total # of data points for each true label
trueLabel_count = frequency_matrix.sum(axis=1, keepdims=True)
# Create Percentage Matrix - number of data points with second best prediction j / total # of data points for true label i
percentage_matrix = np.divide(frequency_matrix, trueLabel_count) * 100
# Turn Matrix Into DataFrame for Visualization
percentage_df = pd.DataFrame(percentage_matrix, index=[f"true_{i}" for i in range(10)], columns=[f"second_{j}" for j in range(10)])
percentage_df

,second_0,second_1,second_2,second_3,second_4,second_5,second_6,second_7,second_8,second_9
true_0,1.217039,0.000000,38.032454,0.709939,0.405680,15.720081,14.503043,3.955375,5.679513,19.776876
true_1,0.088889,0.533333,13.866667,21.955556,0.177778,0.711111,0.622222,32.177778,28.266667,1.600000
true_2,7.307307,13.213213,2.102102,52.652653,4.004004,0.500501,9.109109,5.105105,5.505506,0.500501
true_3,0.686275,6.960784,20.294118,1.568627,0.490196,36.274510,0.294118,4.117647,18.431373,10.882353
true_4,1.025641,0.717949,5.333333,1.743590,2.666667,3.487179,8.307692,18.256410,3.282051,55.179487
true_5,5.210643,2.993348,1.108647,33.592018,4.878049,2.660754,12.527716,0.665188,22.062084,14.301552
true_6,8.859470,0.916497,34.623218,0.203666,21.690428,24.439919,1.018330,0.203666,7.841141,0.203666
true_7,2.879079,1.247601,23.800384,29.174664,5.758157,1.631478,0.095969,1.343570,2.303263,31.765835
true_8,2.564103,6.871795,16.102564,36.102564,2.666667,16.307692,3.282051,1.025641,2.564103,12.512821
true_9,0.402414,0.603622,0.402414,16.800805,46.378270,3.219316,0.000000,24.547284,5.331992,2.313883


Looking at the matrix of we can see that digits that are visually similar have a higher percentage of second-best prediction labels for corresponding true values. For example, for true label 2 the label 3 was had 52% of all second best prediction labels. Or for true label 9 and second best label 4, it was 46% of all second best labels. Vice versa, true label 4 and second best label 9 had 55% of all second best labels. This shows that the MLP we have created has learned a representation that reflects human-perceived digit similarity. When the model is uncertain, the second best prediction is often a digit with a similar shape. This exercise illustrates that even when the best prediction is confident, the second-best prediction distribution shows which digits the model tends to confuse it with.

------------------------------
------------------------------
------------------------------

## <font color='blue'>Section 1: Working with warm start for focused training



This section explores scikit-learn's "warm start" option, which lets training resume from its current weights instead of reinitializing. It trains the network for $k$ epochs, then uses warm start to continue training for another $k$ epochs, with $k$ chosen as a parameter.

In [31]:
# Question 1-0
# Initialize parameter k
k = 5

# Initialize MLP Classifier with Warm Start
mlp_warm = MLPClassifier(verbose = True, warm_start=True, hidden_layer_sizes=(50,), batch_size = 64, \
                          max_iter=k, solver='sgd', activation='logistic',\
                          learning_rate = 'constant', learning_rate_init = 0.001, random_state=2501)
print(f"First Training Run - {k} epochs")
mlp_warm.fit(X_train, y_train)
y_pred_Run1 = mlp_warm.predict(X_test)
acc_Run1 = accuracy_score(y_test, y_pred_Run1)
print(f"Accuracy after First Training Run: {acc_Run1}")

print(f"Second Training Run With Warm Start- {k} epochs")
mlp_warm.fit(X_train, y_train)
y_pred_Run2 = mlp_warm.predict(X_test)
acc_Run2 = accuracy_score(y_test, y_pred_Run2)
print(f"Accuracy after Second Training Run With Warm Start: {acc_Run2}")

First Training Run - 5 epochs
Iteration 1, loss = 1.70994232
Iteration 2, loss = 0.95569378
Iteration 3, loss = 0.67110556
Iteration 4, loss = 0.54074227
Iteration 5, loss = 0.46856456
Accuracy after First Training Run: 0.8924
Second Training Run With Warm Start- 5 epochs


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 1, loss = 0.42353826
Iteration 2, loss = 0.39235466
Iteration 3, loss = 0.36922792
Iteration 4, loss = 0.35147766
Iteration 5, loss = 0.33702011
Accuracy after Second Training Run With Warm Start: 0.9078


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(


When it comes to training, warm start is a setting that doesn't allow the weights to be re-initialized after you call .fit() again. When the warm start setting is set to false and you call .fit(), it re-initializes the weights each time. However, when you set the warm start setting to true, each new .fit() call continues training from the current weights instead. After the first k epochs the test accuracy was about 89%, and after another k epochs of warm-start training it increased to about 91%. This shows that warm starts let us train the models in stages without re-initializing the weights each time. I think this would be useful to do more focused or incremental training without having the cost of starting from random weights again.

To test warm start on a dataset that shifts slightly between training stages, MNIST is randomly split into three parts — train1 (50,000 points), train2 (10,000 points), and test (10,000 points) — with every digit class represented in both train1 and train2.

In [42]:
# Question 1-1
# Get test size of 10K into X_test and y_test - Remaining data points stored in X_temp/y_temp
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=10000, random_state=2501, stratify=y)

# Using X_temp/y_temp - Get Train 1 and Train 2. Treat Train 2 as the test data set for this function call
X_train1, X_train2, y_train1, y_train2 = train_test_split(X_temp, y_temp, test_size=10000, random_state=2501, stratify=y_temp)
# Print out shape of datasets to show they are the right size
print("Train1:", X_train1.shape, y_train1.shape)
print("Train2:", X_train2.shape, y_train2.shape)
print("Test :", X_test.shape, y_test.shape)
# Print out the counts of class labels for each training set
print("Train1 counts:\n", np.bincount(y_train1))
print("\nTrain2 counts:\n", np.bincount(y_train2))

Train1: (50000, 784) (50000,)
Train2: (10000, 784) (10000,)
Test : (10000, 784) (10000,)
Train1 counts:
 [4931 5627 4992 5101 4874 4509 4912 5209 4875 4970]

Train2 counts:
 [ 986 1125  999 1020  975  902  982 1042  975  994]


As you can see from the values of Train 1 Counts and Train 2 Counts, each class label has a non-zero count. That means each class label appears in both training sets.

Using the same $k$ as before, the model is first trained on train1 for $2k$ epochs, then trained further on train2 for another $2k$ epochs using warm start — so each subset gets $2k$ passes, matching the total exposure used in the earlier warm-start example.

In [43]:
# Question 1-2
# Initialize Parameter K - Same as Above
k = 5
# Initialize MLP Classifier - Warm Start & Max_Iter = 2*k
mlp_warm = MLPClassifier(verbose = True, warm_start=True, hidden_layer_sizes=(50,), batch_size = 64, \
                          max_iter=2*k, solver='sgd', activation='logistic',\
                          learning_rate = 'constant', learning_rate_init = 0.001, random_state=2501)

print(f"First Training Run - {2*k} epochs")
mlp_warm.fit(X_train1, y_train1)
y_pred_Run1 = mlp_warm.predict(X_test)
acc_Run1 = accuracy_score(y_test, y_pred_Run1)
print(f"Accuracy after First Training Run: {acc_Run1}")

print(f"Second Training Run With Warm Start- {2*k} epochs")
mlp_warm.fit(X_train2, y_train2)
y_pred_Run2 = mlp_warm.predict(X_test)
acc_Run2 = accuracy_score(y_test, y_pred_Run2)
print(f"Accuracy after Second Training Run With Warm Start: {acc_Run2}")

First Training Run - 10 epochs
Iteration 1, loss = 1.79699550
Iteration 2, loss = 1.07474830
Iteration 3, loss = 0.75967619
Iteration 4, loss = 0.60691607
Iteration 5, loss = 0.52031202
Iteration 6, loss = 0.46553269
Iteration 7, loss = 0.42807498
Iteration 8, loss = 0.40068725
Iteration 9, loss = 0.37959626
Iteration 10, loss = 0.36291841
Accuracy after First Training Run: 0.9051
Second Training Run With Warm Start- 10 epochs


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 1, loss = 0.34423993
Iteration 2, loss = 0.34012702
Iteration 3, loss = 0.33658188
Iteration 4, loss = 0.33337203
Iteration 5, loss = 0.33036153
Iteration 6, loss = 0.32720737
Iteration 7, loss = 0.32429488
Iteration 8, loss = 0.32156434
Iteration 9, loss = 0.31846120
Iteration 10, loss = 0.31579951
Accuracy after Second Training Run With Warm Start: 0.9089


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


After the first 2*k epochs the test accuracy was about 90.5%, and after another 2*k epochs of warm-start training it increased to about 90.9%. The first run was using 50K data points while the second run was only using 10K. I think since the first run had more data points this helped the accuracy after training 2*k epochs, resulting in a 90.5% accuracy. However, the second run did not see a substantial increase or a similar increase like in Question 1-0. I think the reason for this is a decrease in the number of data points used in the second run. By seeing less of the dataset during the second training, it did not learn as much and therefore, did not improve accuracy by a large amount.

This compares the two warm-start strategies above — training on the same dataset twice (Section 1-0) versus training on train1 then fine-tuning on train2 (Section 1-2) — to see which produces better accuracy.

I think Question 1-2's strategy provides slightly better performance. In Q1-2, the model on the first run trains for 2k epochs on the larger part of the training data (train1 - 50K). On the second run (with train2 - 10K), it fine tunes the model without overwriting the earlier knowledge (weights). In Q1-0, the model trains for k epochs both runs using the same dataset each time. Both Q1-0 and Q1-2 beneft from the warm start setting as they both don't have to restart with random weights for the second run. Q1-0 starts at a weaker baseline after the first run compared to Q1-2. Q1-2 also gives a higher final accuracy compared to Q1-0. Q1-0, revisits the same dataset for both runs, which mainly help the model converge. Q1-2 learns on the large subset first then fine tunes on the second subset. That is why the increase between the first and second run is greater in Q1-0 vs Q1-2. This smaller decrease in between first and second run shows the diminishing returns of training on the smaller dataset. However, this may cause mild overfitting to the smaller subset if pushed too far. Overall, I think Q1-2 training for more epochs helps the performance of the dataset. However, I think Q1-0's approach of looking at the entire dataset for each run, can help increase the accuracy gain over each run versus using a subsets for each run.

--------------------------
--------------------------
--------------------------

## <font color='blue'> Section 2. How Powerful are Neural Networks? </font>

There is a lot of hype surrounding Neural Networks, and their ability to solve problems. This is of course justified, but it's worth being aware of their limitations.

To test whether the MLP can exploit shortcuts, the label of each image is "hidden" inside the image itself: the top-left pixel (originally a value from 0-255, scaled to 0-1) is overwritten with the image's own label divided by 255. Every row of $X$ corresponds to one image, with each column holding one grayscale pixel value.

A new dataset $X1$ is built by copying $X$ and overwriting its first column with each row's true label (divided by 255), then splitting $X1$ into training and test sets the same way as $X$.

In [52]:
# Question 2-0
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
y = y.astype(int)
X = ((X / 255.) - .5) * 2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=10000, random_state=2501, stratify=y)
# Copy X into X1
X1 = X.copy()

# Replace first pixel with normalized label - first pixel = first entry of each row; normalized label = label / 255
X1[:, 0] = y / 255.0

# Split dataset into test and train sets
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y, test_size=10000, random_state=2501, stratify=y)

[5. 0. 4. ... 4. 5. 6.]
[5 0 4 ... 4 5 6]


In principle, a network could reach 100% accuracy on $X1$ by learning to ignore every pixel except the planted label — setting all other weights to zero. This section tests whether an MLP with a single hidden layer actually discovers that shortcut during training, by training it on $X1$ and comparing its accuracy against the model trained on the original, unmodified $X$.

In [51]:
# Question 2-1
mlp_original = MLPClassifier(verbose = True, hidden_layer_sizes=(50,), batch_size = 64, \
                          max_iter=500, solver='sgd', activation='logistic',\
                          learning_rate = 'constant', learning_rate_init = 0.001, random_state=2501)

mlp_transformed = MLPClassifier(verbose = True, hidden_layer_sizes=(50,), batch_size = 64, \
                          max_iter=500, solver='sgd', activation='logistic',\
                          learning_rate = 'constant', learning_rate_init = 0.001, random_state=2501)

mlp_original.fit(X_train, y_train)
y_pred_orig = mlp_original.predict(X_test)
acc_orig = accuracy_score(y_test, y_pred_orig)
print(f"Test accuracy on original X: {acc_orig:.4f}")

mlp_transformed.fit(X1_train, y1_train)
y_pred_transformed = mlp_transformed.predict(X1_test)
acc_transformed = accuracy_score(y1_test, y_pred_transformed)
print(f"Test accuracy on transformed X1: {acc_transformed:.4f}")

Iteration 1, loss = 1.70994232
Iteration 2, loss = 0.95569378
Iteration 3, loss = 0.67110556
Iteration 4, loss = 0.54074227
Iteration 5, loss = 0.46856456
Iteration 6, loss = 0.42324863
Iteration 7, loss = 0.39204168
Iteration 8, loss = 0.36907797
Iteration 9, loss = 0.35130105
Iteration 10, loss = 0.33671814
Iteration 11, loss = 0.32456279
Iteration 12, loss = 0.31429788
Iteration 13, loss = 0.30533321
Iteration 14, loss = 0.29724721
Iteration 15, loss = 0.28987095
Iteration 16, loss = 0.28326889
Iteration 17, loss = 0.27717208
Iteration 18, loss = 0.27166676
Iteration 19, loss = 0.26639564
Iteration 20, loss = 0.26140157
Iteration 21, loss = 0.25671940
Iteration 22, loss = 0.25234116
Iteration 23, loss = 0.24817009
Iteration 24, loss = 0.24429060
Iteration 25, loss = 0.24042280
Iteration 26, loss = 0.23676118
Iteration 27, loss = 0.23328458
Iteration 28, loss = 0.23005543
Iteration 29, loss = 0.22673805
Iteration 30, loss = 0.22369456
Iteration 31, loss = 0.22066047
Iteration 32, los

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Test accuracy on transformed X1: 0.9702


With a single layer of 50 neurons and max_iter of 50, we achieved a accuracy of X on the original dataset 0.9713, and a accuracy of 0.9702 on the transformed dataset X1.The model of the original dataset was able to converge, unlike the transformed dataset that hit the max_iter of 500. The accuracy of the transformed model does not reach 100% and is approximately the same as the accuracy of the original model. This shows that the MLP classifier does not learn the shortcut in the data: rely only on the first pixel for the label. With the transformed data model also not converging in the same number or less epochs, the original data seems to be the better choice.